In [8]:
import os
import sys

PROJECT_ROOT = r"C:\Users\Admin\NEIRO\SPR_2_FIN_PROJ"

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
SRC_DIR = os.path.join(PROJECT_ROOT, "src")

model_path = os.path.join(PROJECT_ROOT, "models", "birnn_lstm_lm.pth")
tokenizer_path = os.path.join(PROJECT_ROOT, "data", "processed", "tokenizer")
processed_texts_path = os.path.join(PROJECT_ROOT, "data", "dataset_processed.txt")
datasets_path = os.path.join(PROJECT_ROOT, "data", "processed")
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)


In [9]:
#подготовка датасета 

import sys
import os

# Получаем текущую рабочую директорию (где открыт ноутбук)
notebook_dir = os.getcwd()

import re
import pandas as pd
from tqdm import tqdm
import os

def clean_tweet(text):
    """
    Очистка и нормализация текста твита
    """
    if not isinstance(text, str):
        return ""
    
    # 1. Привести к нижнему регистру
    text = text.lower()
    
    # 2. Удалить ссылки
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # 3. Удалить упоминания пользователей
    text = re.sub(r'@\w+', '', text)
    
    # 4. Удалить специальные символы и цифры (опционально)
    # Оставляем только буквы и пробелы
    text = re.sub(r'[^a-z\s]', ' ', text)

    # 4.5 Удалить хэштеги и символ # (важное исправление!)
    text = re.sub(r'#', '', text)
    
    # 5. Удалить эмодзи и специальные символы Unicode
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    
    # 6. Заменить множественные пробелы одним
    text = re.sub(r'\s+', ' ', text)
    
    # 7. Удалить пробелы в начале и конце
    text = text.strip()
    
    return text

def process_tweets_file(input_file, output_file, max_tweets=100000, random_sample=False):

    print(f"Чтение файла {input_file}...")
    
    if random_sample:
        # Читаем все строки для случайной выборки
        with open(input_file, 'r', encoding='utf-8', errors='ignore') as f:
            all_tweets = f.readlines()
        
        print(f"Всего твитов в файле: {len(all_tweets)}")
        
        if len(all_tweets) > max_tweets:
            tweets = random.sample(all_tweets, max_tweets)
            print(f"Взята случайная выборка из {max_tweets} твитов")
        else:
            tweets = all_tweets
            print(f"Используются все {len(tweets)} твитов")
    else:
        # Читаем только первые N строк
        tweets = []
        with open(input_file, 'r', encoding='utf-8', errors='ignore') as f:
            for i, line in enumerate(f):
                if i >= max_tweets:
                    break
                tweets.append(line)
        
        print(f"Загружено {len(tweets)} твитов (первые {max_tweets})")
    
    processed_texts = []
    
    # Обработка каждого твита с прогресс-баром
    for tweet in tqdm(tweets, desc="Обработка твитов"):
        cleaned_text = clean_tweet(tweet)
        
        if cleaned_text and len(cleaned_text) > 3:
            processed_texts.append(cleaned_text)
    
    # Сохранение
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    with open(output_file, 'w', encoding='utf-8') as f:
        for text in processed_texts:
            f.write(text + '\n')
    
    print(f"\nСтатистика:")
    print(f"Обработано твитов: {len(processed_texts)}")
    print(f"Сохранено в: {output_file}")
    
    return processed_texts

# Основная часть скрипта
if __name__ == "__main__":
    
    # Формируем пути
    input_file = os.path.join(PROJECT_ROOT, "data", "tweets.txt")
    output_file = os.path.join(PROJECT_ROOT, "data", "dataset_processed.txt")  # Меняем на .txt

    print(f"Путь к исходному файлу: {input_file}")
    print(f"Путь к выходному файлу: {output_file}")
    
    # Запуск обработки
    try:
        processed_texts = process_tweets_file(input_file, output_file)
        print("Обработка завершена успешно!")
        
    except FileNotFoundError:
        print(f"Ошибка: Файл {input_file} не найден.")
        print("Пожалуйста, убедитесь, что файл находится в той же директории.")
    except Exception as e:
        print(f"Произошла ошибка: {e}")

Путь к исходному файлу: C:\Users\Admin\NEIRO\SPR_2_FIN_PROJ\data\tweets.txt
Путь к выходному файлу: C:\Users\Admin\NEIRO\SPR_2_FIN_PROJ\data\dataset_processed.txt
Чтение файла C:\Users\Admin\NEIRO\SPR_2_FIN_PROJ\data\tweets.txt...
Загружено 100000 твитов (первые 100000)


Обработка твитов: 100%|██████████| 100000/100000 [00:01<00:00, 83694.42it/s]


Статистика:
Обработано твитов: 99716
Сохранено в: C:\Users\Admin\NEIRO\SPR_2_FIN_PROJ\data\dataset_processed.txt
Обработка завершена успешно!


In [10]:
#DATASET PREPARE
# импортируем библиотеки, которые пригодятся для задачи
import torch
import os
import torch.nn as nn
import pandas as pd  # Добавляем импорт pandas
import re
import random
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast
from transformers import GPT2Tokenizer  # Меняем на GPT-2
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# Пути к файлам
# script_dir = os.path.dirname(os.path.abspath(__file__))
# project_root = os.path.dirname(script_dir)
# data_path = os.path.join(project_root, "data", "dataset_processed.txt")
random.seed(42)
torch.manual_seed(42)

# Загружаем данные
if processed_texts_path.endswith('.csv'):
    df = pd.read_csv(processed_texts_path)
    dataset = df['cleaned_text'].tolist()
else:
    with open(processed_texts_path, 'r', encoding='utf-8') as f:
        dataset = [line.strip() for line in f]

print(f"Загружено {len(dataset)} текстов")

seq_len = 7

# удаляем слишком короткие тексты
cleaned_texts = [line for line in dataset if len(line.split()) >= seq_len]

print(f"После фильтрации осталось {len(cleaned_texts)} текстов")

# для упрощения используем только max_texts_count текстов
max_texts_count = 170000

# Разделяем данные на train/val/test
test_size = 0.1  # 10% на тест
val_size = 0.1   # 10% на валидацию от оставшихся после теста

print(f"\nРазделение данных на train/val/test...")

# Сначала разделим на train+val и test
train_val_texts, test_texts = train_test_split(
    cleaned_texts[:max_texts_count], 
    test_size=test_size, 
    random_state=42
)

# Затем train_val разделим на train и val
val_ratio = val_size / (1 - test_size)  # val_size от train_val_texts
train_texts, val_texts = train_test_split(
    train_val_texts, 
    test_size=val_ratio, 
    random_state=42
)

print(f"Размеры выборок:")
print(f"  Train: {len(train_texts)} текстов")
print(f"  Val: {len(val_texts)} текстов")  
print(f"  Test: {len(test_texts)} текстов")
print(f"  Всего: {len(train_texts) + len(val_texts) + len(test_texts)} текстов")

# класс датасета
class NextTokenDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=512):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.samples = []
        self.original_texts = []  # Сохраняем оригинальные тексты
        self.sample_to_text_idx = []  # Сохраняем индекс текста для каждого примера

        print("Токенизация текстов...")
        for text_idx, line in enumerate(tqdm(texts)):
            token_ids = tokenizer.encode(line, add_special_tokens=True, max_length=self.max_len, truncation=True)
            # Создаем пары для каждого токена в последовательности
            if len(token_ids) < 2:
                continue
            input_ids = token_ids[:-1]
            target_ids = token_ids[1:]
            self.samples.append((input_ids, target_ids))
            self.original_texts.append(line)  # Сохраняем оригинальный текст
            self.sample_to_text_idx.append(text_idx)  # Сохраняем индекс текста
            # for i in range(1, len(token_ids) - 1):
            #     context = token_ids[1:i+1]
            #     target = token_ids[i+1]
            #     self.samples.append((context, target))
            #     self.original_texts.append(line)  # Сохраняем оригинальный текст
            #     self.sample_to_text_idx.append(text_idx)  # Сохраняем индекс текста
           
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.tensor(x), torch.tensor(y)
    
    def get_original_text(self, idx):
        """Возвращает оригинальный текст для примера с индексом idx"""
        return self.original_texts[idx]
    
    def get_text_index(self, idx):
        """Возвращает индекс текста в исходном списке"""
        return self.sample_to_text_idx[idx]

def collate_fn(batch):
    xs, ys = zip(*batch)

    xs = torch.nn.utils.rnn.pad_sequence(xs, batch_first=True, padding_value=0)
    ys = torch.nn.utils.rnn.pad_sequence(ys, batch_first=True, padding_value=0)

    return xs, ys




if __name__ == "__main__":
    # загружаем токенизатор
    tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
    # Дополнительная функция для сохранения данных
    def save_datasets(train_texts, val_texts, test_texts, tokenizer, save_dir="data/processed"):
        """Сохраняет разделенные данные и токенизатор"""
        # os.makedirs(save_dir, exist_ok=True)
        os.makedirs(datasets_path, exist_ok=True)
        
        # Сохраняем тексты
        with open(os.path.join(datasets_path, "train.txt"), "w", encoding="utf-8") as f:
            f.write("\n".join(train_texts))
        
        with open(os.path.join(datasets_path, "val.txt"), "w", encoding="utf-8") as f:
            f.write("\n".join(val_texts))
            
        with open(os.path.join(datasets_path, "test.txt"), "w", encoding="utf-8") as f:
            f.write("\n".join(test_texts))
        
        # Сохраняем токенизатор
        tokenizer.save_pretrained(os.path.join(datasets_path, "tokenizer"))
        
        print(f"\nДанные сохранены в папку: {datasets_path}")
        print(f"  train.txt: {len(train_texts)} текстов")
        print(f"  val.txt: {len(val_texts)} текстов")
        print(f"  test.txt: {len(test_texts)} текстов")

    # Сохраняем данные для использования в других файлах
    save_datasets(train_texts, val_texts, test_texts, tokenizer)


    # тренировочный, валидационный и тестовый датасеты
    print("\nСоздание тренировочного датасета...")
    train_dataset = NextTokenDataset(train_texts, tokenizer)

    print("\nСоздание валидационного датасета...")
    val_dataset = NextTokenDataset(val_texts, tokenizer)

    print("\nСоздание тестового датасета...")
    test_dataset = NextTokenDataset(test_texts, tokenizer)

    print("DONE")
    print(f"\nDataLoader'ы созданы:")
    print(f"  Train: {len(train_dataset)} примеров")
    print(f"  Val: {len(val_dataset)} примеров")
    print(f"  Test: {len(test_dataset)} примеров")
    print(f"  Примерное число примеров на текст: {len(train_dataset) / len(train_texts):.1f}")

    # даталоадеры
    train_loader = DataLoader(
        train_dataset, 
        batch_size=64, 
        shuffle=True,
        collate_fn=collate_fn
    )

    val_loader = DataLoader(
        val_dataset, 
        batch_size=64,
        collate_fn=collate_fn
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=64,
        collate_fn=collate_fn
    )



    # Демонстрация работы
    print("\n" + "=" * 60)
    print("ДЕМОНСТРАЦИЯ РАБОТЫ:")
    print("=" * 60)

    # Получаем первый батч из тренировочного датасета
    batch = next(iter(train_loader))
    x_batch, y_batch = batch

    print(f"\nРазмеры тренировочного батча:")
    print(f"  X (context): {x_batch.shape}")
    print(f"  Y (target): {y_batch.shape}")

    # Проверяем работу на тестовом примере
    print("\n" + "=" * 60)
    print("ТЕСТОВАЯ ПРОВЕРКА ЛОГИКИ:")
    print("=" * 60)

    test_text = "Hello world this is a test"
    print(f"\nТестовый текст: '{test_text}'")

    test_tokens = tokenizer.encode(test_text, add_special_tokens=True)
    print(f"Токены с спецсимволами: {tokenizer.convert_ids_to_tokens(test_tokens)}")

    test_texts = [test_text]
    test_small_dataset = NextTokenDataset(test_texts, tokenizer)

    print(f"\nВсе примеры из этого текста ({len(test_small_dataset)} примеров):")

    for i in range(min(7, len(test_small_dataset))):
        x, y = test_small_dataset[i]
        
        print(f"\nПример {i}:")
        # print(f"  X (контекст): {tokenizer.convert_ids_to_tokens(x)}")
        # print(f"  Y (цель): {tokenizer.convert_ids_to_tokens([y.item()])[0]}")
        x_tokens = tokenizer.convert_ids_to_tokens(x.tolist())
        y_tokens = tokenizer.convert_ids_to_tokens(y.tolist())

        print("Контекст → цель:")
        print(f"{x_tokens[-1]} → {y_tokens[-1]}")

        
    print("\n" + "=" * 60)
    print("ПОДГОТОВКА ДАННЫХ ЗАВЕРШЕНА УСПЕШНО!")
    print("=" * 60)

Загружено 99716 текстов
После фильтрации осталось 80083 текстов

Разделение данных на train/val/test...
Размеры выборок:
  Train: 64065 текстов
  Val: 8009 текстов
  Test: 8009 текстов
  Всего: 80083 текстов

Данные сохранены в папку: C:\Users\Admin\NEIRO\SPR_2_FIN_PROJ\data\processed
  train.txt: 64065 текстов
  val.txt: 8009 текстов
  test.txt: 8009 текстов

Создание тренировочного датасета...
Токенизация текстов...


100%|██████████| 64065/64065 [00:08<00:00, 7561.99it/s]



Создание валидационного датасета...
Токенизация текстов...


100%|██████████| 8009/8009 [00:01<00:00, 7640.53it/s]



Создание тестового датасета...
Токенизация текстов...


100%|██████████| 8009/8009 [00:01<00:00, 7648.02it/s]


DONE

DataLoader'ы созданы:
  Train: 64065 примеров
  Val: 8009 примеров
  Test: 8009 примеров
  Примерное число примеров на текст: 1.0

ДЕМОНСТРАЦИЯ РАБОТЫ:

Размеры тренировочного батча:
  X (context): torch.Size([64, 35])
  Y (target): torch.Size([64, 35])

ТЕСТОВАЯ ПРОВЕРКА ЛОГИКИ:

Тестовый текст: 'Hello world this is a test'
Токены с спецсимволами: ['[CLS]', 'hello', 'world', 'this', 'is', 'a', 'test', '[SEP]']
Токенизация текстов...


100%|██████████| 1/1 [00:00<?, ?it/s]


Все примеры из этого текста (1 примеров):

Пример 0:
Контекст → цель:
test → [SEP]

ПОДГОТОВКА ДАННЫХ ЗАВЕРШЕНА УСПЕШНО!


In [12]:
import torch
import torch.nn as nn
import os
from tqdm import tqdm


class BiRNNClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=128, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            emb_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, input_ids, hidden=None):

        emb = self.embedding(input_ids)
        out, hidden = self.lstm(emb, hidden)
        logits = self.fc(out)
        return logits, hidden
    
    def predict_next_token(self, input_ids, temperature=1.0, device="cpu"):
        self.eval()

        with torch.no_grad():
            if isinstance(input_ids, list):
                input_ids = torch.tensor([input_ids], device=device)
            elif input_ids.dim() == 1:
                input_ids = input_ids.unsqueeze(0).to(device)
            else:
                input_ids = input_ids.to(device)

            logits, _ = self(input_ids)

            # берём логиты последнего токена
            logits = logits[:, -1, :] / temperature

            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            return next_token.item()
    
    def generate_text(self, input_ids, max_new_tokens=50, temperature=1.0, device="cpu", tokenizer=None):
        self.eval()

        with torch.no_grad():
            if isinstance(input_ids, list):
                generated = input_ids.copy()
            else:
                generated = input_ids.tolist()

            for _ in range(max_new_tokens):
                next_token = self.predict_next_token(
                    generated,
                    temperature=temperature,
                    device=device
                )

                # стоп-токен
                if tokenizer and next_token == tokenizer.sep_token_id:
                    break

                generated.append(next_token)

                # ограничение контекста
                if len(generated) > 256:
                    generated = generated[-256:]

            return generated




def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

if __name__ == "__main__":
    from transformers import BertTokenizerFast
    import os
    
    # Определяем путь к токенизатору
    # script_dir = os.path.dirname(os.path.abspath(__file__))
    # project_root = os.path.dirname(script_dir)
    # tokenizer_path = os.path.join(project_root, "data", "processed", "tokenizer")
    
    # Загружаем токенизатор
    tokenizer = BertTokenizerFast.from_pretrained(tokenizer_path)
    vocab_size = tokenizer.vocab_size
    
    print(f"Размер словаря: {vocab_size}")
    hidden_dim = 128

    rnn_types = ["RNN", "GRU", "LSTM"]
    combine_methods = ["sum", "concat"]


    # Сравнение
    print(f"{'RNN Type':<8} | {'Combine':<6} | {'Params':>10}")
    print("-" * 35)
    for rnn_type in rnn_types:
        for combine in combine_methods:
            model = BiRNNClassifier(vocab_size=vocab_size, emb_dim=128,hidden_dim=128)
            param_count = count_parameters(model)
            print(f"{rnn_type:<8} | {combine:<6} | {param_count:>10,}") 

Размер словаря: 30522
RNN Type | Combine |     Params
-----------------------------------
RNN      | sum    |  7,976,250
RNN      | concat |  7,976,250
GRU      | sum    |  7,976,250
GRU      | concat |  7,976,250
LSTM     | sum    |  7,976,250
LSTM     | concat |  7,976,250


In [ ]:
#train model

import torch
import torch.nn as nn
import os
from tqdm import tqdm
from torch.utils.data import DataLoader
from transformers import BertTokenizerFast
import matplotlib.pyplot as plt

from lstm_model import BiRNNClassifier
from next_token_dataset import NextTokenDataset, collate_fn


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for x, y in tqdm(loader, desc="Training"):
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits, _ = model(x)

        loss = criterion(
            logits.view(-1, logits.size(-1)),
            y.view(-1)
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    for x, y in tqdm(loader, desc="Evaluating"):
        x = x.to(device)
        y = y.to(device)

        logits, _ = model(x)

        loss = criterion(
            logits.view(-1, logits.size(-1)),
            y.view(-1)
        )

        total_loss += loss.item()

    return total_loss / len(loader)


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    # пути
    # data_dir = "data/processed"
    # tokenizer = BertTokenizerFast.from_pretrained(os.path.join(data_dir, "tokenizer"))
    # vocab_size = tokenizer.vocab_size

    # данные
    def load_texts(path):
        with open(path, encoding="utf-8") as f:
            return [l.strip() for l in f if l.strip()]

    train_texts = load_texts(datasets_path + "/train.txt")
    val_texts   = load_texts(datasets_path + "/val.txt")

    train_ds = NextTokenDataset(train_texts, tokenizer)
    val_ds   = NextTokenDataset(val_texts, tokenizer)

    train_loader = DataLoader(
        train_ds,
        batch_size=64,
        shuffle=True,
        collate_fn=collate_fn
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=64,
        shuffle=False,
        collate_fn=collate_fn
    )

    # модель
    model = BiRNNClassifier(
        vocab_size=vocab_size,
        emb_dim=128,
        hidden_dim=128
    ).to(device)

    print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    criterion = nn.CrossEntropyLoss(
        ignore_index=tokenizer.pad_token_id
    )

    n_epochs = 3
    train_losses = []
    val_losses = []


    for epoch in range(n_epochs):
        train_loss = train_epoch(
            model, train_loader, optimizer, criterion, device
        )
        val_loss = eval_epoch(
            model, val_loader, criterion, device
        )

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        print(
            f"Epoch {epoch+1}/{n_epochs} | "
            f"Train loss: {train_loss:.4f} | "
            f"Val loss: {val_loss:.4f}"
        )


    # сохранение
    # os.makedirs("models", exist_ok=True)
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "vocab_size": vocab_size,
            "emb_dim": 128,
            "hidden_dim": 128,
        },
        # "models/birnn_lstm_lm.pth"
        model_path
    )

    print("Mодель сохранена : models/birnn_lstm_lm.pth")


if __name__ == "__main__":
    main()


Device: cpu
Токенизация текстов...


 66%|██████▌   | 42220/64065 [00:04<00:02, 9354.71it/s]

In [ ]:
#Оценка LSTM модели на задаче генерации текста с использованием ROUGE метрики

import sys
import os

# Получаем текущую рабочую директорию (где открыт ноутбук)
notebook_dir = os.getcwd()

# Путь к папке src, где лежит lstm_model.py
src_path = os.path.join(notebook_dir, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

from lstm_model import BiRNNClassifier


import torch
from transformers import BertTokenizerFast
from lstm_model import BiRNNClassifier
import os
from tqdm import tqdm
import evaluate
import numpy as np

# Загружаем метрику ROUGE
rouge_metric = evaluate.load("rouge")


def generate_and_evaluate(model, tokenizer, text, device):

    tokens = tokenizer.encode(text, add_special_tokens=True)
    if len(tokens) < 2:
        return None

    # Контекст для генерации: 3/4 текста
    context_len = int(len(tokens) * 0.75)
    context_one_token = tokens[:context_len]
    target_quarter = tokens[context_len:]

    #  Определяем target_one как первый «реальный» токен из оставшейся четверти
    target_one_token = None
    for tok in target_quarter:
        if tok not in {tokenizer.sep_token_id, tokenizer.cls_token_id, tokenizer.pad_token_id, tokenizer.cls_token_id}:
            target_one_token = [tok]
            break
    if target_one_token is None:
        target_one_token = []

    # Генерация одного токена
    generated_one = model.generate_text(
        context_one_token,
        max_new_tokens=1,
        temperature=1.0,
        device=device,
        tokenizer=tokenizer
    )
    generated_one_token = generated_one[-1:]  # последний токен

    # Генерация 1/4 текста, начиная с сгенерированного токена
    # context_quarter = context_one_token + generated_one_token
    remaining_len = len(target_quarter)  
    generated_quarter = model.generate_text(
        context_one_token,
        max_new_tokens=remaining_len,
        temperature=1,
        device=device,
        tokenizer=tokenizer
    )
    # Берём только сгенерированное продолжение четверти текста
    generated_quarter_only = generated_one_token + generated_quarter[len(context_one_token):]

    # Декодирование
    original_text = tokenizer.decode(tokens, skip_special_tokens=True)
    context_text = tokenizer.decode(context_one_token, skip_special_tokens=True)
    gen_one_text = tokenizer.decode(generated_one_token, skip_special_tokens=True)
    gen_quarter_text = tokenizer.decode(generated_quarter_only, skip_special_tokens=True)
    target_one_text = tokenizer.decode(target_one_token, skip_special_tokens=True)
    target_quarter_text = tokenizer.decode(target_quarter, skip_special_tokens=True)

    # ROUGE
    rouge_one = rouge_metric.compute(predictions=[gen_one_text], references=[target_one_text])
    rouge_quarter = rouge_metric.compute(predictions=[gen_quarter_text], references=[target_quarter_text])

    return {
        "original": original_text,
        "context": context_text,
        "generated_one": gen_one_text,
        "generated_quarter": gen_quarter_text,
        "target_one": target_one_text,
        "target_quarter": target_quarter_text,
        "rouge_one": rouge_one,
        "rouge_quarter": rouge_quarter
    }



def evaluate_texts(model, tokenizer, texts, device, print_examples=True):
    rouge_one_scores = []
    rouge_quarter_scores = []

    # Печать первых 10 примеров
    if print_examples:
        print("\n=== ПЕРВЫЕ 10 ТЕСТОВЫХ ПРИМЕРОВ ===")
        for i, text in enumerate(texts[20:30]):
            res = generate_and_evaluate(model, tokenizer, text, device)
            if res is None:
                continue

            print(f"\nПример {i+1}:")
            print(f"Оригинальный текст: {res['original']}")
            print(f"Промт (3/4 текста): {res['context']}")
            print(f"Сгенерировано один токен: {res['generated_one']} (правильный: {res['target_one']})")
            print(f"Сгенерировано 1/4 текста: {res['generated_quarter']} (правильное продолжение: {res['target_quarter']})")
            print(f"ROUGE-1 один токен: {res['rouge_one']['rouge1']:.4f}")
            print(f"ROUGE-1 1/4 текста: {res['rouge_quarter']['rouge1']:.4f}")
            print("-" * 80)

    # Усреднение ROUGE по всем текстам
    for text in tqdm(texts[:210], desc="Calculating average ROUGE"):
        res = generate_and_evaluate(model, tokenizer, text, device)
        if res is None:
            continue
        rouge_one_scores.append(res['rouge_one']['rouge1'])
        rouge_quarter_scores.append(res['rouge_quarter']['rouge1'])

    avg_rouge_one = np.mean(rouge_one_scores) if rouge_one_scores else 0.0
    avg_rouge_quarter = np.mean(rouge_quarter_scores) if rouge_quarter_scores else 0.0

    print("\n=== СРЕДНИЙ ROUGE ===")
    print(f"Средний ROUGE-1 для одного токена: {avg_rouge_one:.4f}")
    print(f"Средний ROUGE-1 для 1/4 текста: {avg_rouge_quarter:.4f}")


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Путь к модели и токенизатору
    model_path = "models/birnn_lstm_lm.pth"
    tokenizer_path = "data/processed/tokenizer"

    # Загружаем токенизатор
    tokenizer = BertTokenizerFast.from_pretrained(tokenizer_path)

    # Загружаем модель
    checkpoint = torch.load(model_path, map_location=device)
    model = BiRNNClassifier(
        vocab_size=checkpoint['vocab_size'],
        hidden_dim=checkpoint['hidden_dim']
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()

    # Загружаем тестовые тексты
    project_root = r"C:\Users\Admin\NEIRO\SPR_2_FIN_PROJ\src"

    # Путь к файлу с тестами
    data_dir = os.path.join(project_root, "data", "processed")
    test_file = os.path.join(data_dir, "test.txt")
    with open(test_file, "r", encoding="utf-8") as f:
        test_texts = [line.strip() for line in f if line.strip()]

    # Вызываем функцию оценки
    evaluate_texts(model, tokenizer, test_texts, device, print_examples=True)


if __name__ == "__main__":
    main()



=== ПЕРВЫЕ 10 ТЕСТОВЫХ ПРИМЕРОВ ===

Пример 1:
Оригинальный текст: sigh am sitting here working with my leg propped up it s making my ankle feel better but also making my knee hurt
Промт (3/4 текста): sigh am sitting here working with my leg propped up it s making my ankle feel better
Сгенерировано один токен: practice (правильный: but)
Сгенерировано 1/4 текста: practice (правильное продолжение: but also making my knee hurt)
ROUGE-1 один токен: 0.0000
ROUGE-1 1/4 текста: 0.0000
--------------------------------------------------------------------------------

Пример 2:
Оригинальный текст: feeling really sick today how about you
Промт (3/4 текста): feeling really sick today how
Сгенерировано один токен: cold (правильный: about)
Сгенерировано 1/4 текста: cold you did talking (правильное продолжение: about you)
ROUGE-1 один токен: 0.0000
ROUGE-1 1/4 текста: 0.3333
--------------------------------------------------------------------------------

Пример 3:
Оригинальный текст: bored amp tire

Calculating average ROUGE: 100%|██████████| 210/210 [01:09<00:00,  3.03it/s]


=== СРЕДНИЙ ROUGE ===
Средний ROUGE-1 для одного токена: 0.0857
Средний ROUGE-1 для 1/4 текста: 0.0354


In [ ]:
#оценка трансформерной модели на задаче генерации текста с использованием ROUGE метрики GPT2 вместо distilgpt2, так как distilgpt2 давал точность оклоло 2% что меньше чем LSTM модель

import os
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import evaluate
import numpy as np
from tqdm import tqdm


import sys
import os

# Получаем текущую рабочую директорию (где открыт ноутбук)
notebook_dir = os.getcwd()

# Путь к папке src, где лежит lstm_model.py
src_path = os.path.join(notebook_dir, "src")

# Загружаем тестовые тексты
project_root = r"C:\Users\Admin\NEIRO\SPR_2_FIN_PROJ\src"

# Путь к файлу с тестами
data_dir = os.path.join(project_root, "data", "processed")
test_file = os.path.join(data_dir, "test.txt")
# --- Функция для загрузки тестовых текстов ---
def load_texts(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

# Загружаем тестовые тексты
project_root = r"C:\Users\Admin\NEIRO\SPR_2_FIN_PROJ\src"

# Путь к файлу с тестами
data_dir = os.path.join(project_root, "data", "processed")
test_file = os.path.join(data_dir, "test.txt")
with open(test_file, "r", encoding="utf-8") as f:
    test_texts = [line.strip() for line in f if line.strip()]

# --- Загрузка модели и токенизатора ---
model_name = "GPT2" # на модели distilgpt2 rouge около 2% то есть меньше чем на LSTM поэтому переделал на GPT2
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# --- Pipeline для генерации ---
generator = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    device=-1,  # CPU; для GPU ставьте 0
)

# --- Метрика ROUGE ---
rouge = evaluate.load("rouge")

# --- Функция для генерации и оценки ---
def generate_and_evaluate_transformer(text):
    words = text.split()
    if len(words) < 5 or len(words) > 100:
        return None

    # Делим на 3/4 и 1/4
    split_idx = int(len(words) * 0.75)
    prompt_text = ' '.join(words[:split_idx])
    target_text = ' '.join(words[split_idx:])

    # Генерация продолжения
    out = generator(
        prompt_text,
        max_new_tokens=len(words) - split_idx,
        num_return_sequences=1,
        do_sample=False,
        top_p=0.95,
        temperature=1
    )

    generated_full = out[0]["generated_text"]
    # Берём только сгенерированное продолжение
    if generated_full.startswith(prompt_text):
        generated_part = generated_full[len(prompt_text):].strip()
    else:
        generated_part = generated_full

    # ROUGE
    rouge_scores = rouge.compute(predictions=[generated_part], references=[target_text])

    return {
        "original": text,
        "prompt": prompt_text,
        "generated_quarter": generated_part[:100],
        "target_quarter": target_text,
        "rouge1": rouge_scores["rouge1"],
        "rouge2": rouge_scores["rouge2"],
        "rougeL": rouge_scores["rougeL"]
    }

#  Вывод первых 10 примеров ---
print("\n=== ПЕРВЫЕ 10 ПРИМЕРОВ ===")
for i, text in tqdm(enumerate(test_texts[:10])):
    res = generate_and_evaluate_transformer(text)
    if res is None:
        continue

    print(f"\nПример {i+1}:")
    print(f"Оригинальный текст: {res['original']}")
    print(f"Промт (3/4 текста): {res['prompt']}")
    print(f"Сгенерировано 1/4 текста: {res['generated_quarter']}")
    print(f"Правильное продолжение: {res['target_quarter']}")
    print(f"ROUGE-1: {res['rouge1']:.4f}")
    print(f"ROUGE-2: {res['rouge2']:.4f}")
    print(f"ROUGE-L: {res['rougeL']:.4f}")
    print("-" * 80)

#  Средний ROUGE по 100 тестовым текстам ---
rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for text in tqdm(test_texts[:100], desc="Вычисление среднего ROUGE"):
    res = generate_and_evaluate_transformer(text)
    if res is None:
        continue
    rouge1_scores.append(res['rouge1'])
    rouge2_scores.append(res['rouge2'])
    rougeL_scores.append(res['rougeL'])

avg_rouge1 = np.mean(rouge1_scores) if rouge1_scores else 0.0
avg_rouge2 = np.mean(rouge2_scores) if rouge2_scores else 0.0
avg_rougeL = np.mean(rougeL_scores) if rougeL_scores else 0.0

print("\n=== СРЕДНИЙ ROUGE ПО 100 ТЕСТАМ ===")
print(f"Средний ROUGE-1: {avg_rouge1:.4f}")
print(f"Средний ROUGE-2: {avg_rouge2:.4f}")
print(f"Средний ROUGE-L: {avg_rougeL:.4f}")


Device set to use cpu



=== ПЕРВЫЕ 10 ПРИМЕРОВ ===


0it [00:00, ?it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
1it [00:00,  2.12it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Пример 1:
Оригинальный текст: she turn out fine at least she didn t annoy me doing malay hw at pm i don t know for what reason but i kinda miss hannah
Промт (3/4 текста): she turn out fine at least she didn t annoy me doing malay hw at pm i don t know for
Сгенерировано 1/4 текста: sure but i think she was just
Правильное продолжение: what reason but i kinda miss hannah
ROUGE-1: 0.2857
ROUGE-2: 0.1667
ROUGE-L: 0.2857
--------------------------------------------------------------------------------


2it [00:00,  2.99it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


...


Пример 9:
Оригинальный текст: i dreamt that i was failing one of my classes it was not fun
Промт (3/4 текста): i dreamt that i was failing one of my classes
Сгенерировано 1/4 текста: . I was so
Правильное продолжение: it was not fun
ROUGE-1: 0.2857
ROUGE-2: 0.0000
ROUGE-L: 0.2857
--------------------------------------------------------------------------------


10it [00:02,  3.72it/s]



Пример 10:
Оригинальный текст: thanks i ll need it i hate packing
Промт (3/4 текста): thanks i ll need it i
Сгенерировано 1/4 текста: will be
Правильное продолжение: hate packing
ROUGE-1: 0.0000
ROUGE-2: 0.0000
ROUGE-L: 0.0000
--------------------------------------------------------------------------------


Вычисление среднего ROUGE:   0%|          | 0/100 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Вычисление среднего ROUGE:   1%|          | 1/100 [00:00<00:32,  3.01it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Вычисление среднего ROUGE:   2%|▏         | 2/100 [00:00<00:27,  3.58it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Вычисление среднего ROUGE:   3%|▎         | 3/100 [00:00<00:28,  3.39it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` fo


=== СРЕДНИЙ ROUGE ПО 100 ТЕСТАМ ===
Средний ROUGE-1: 0.1213
Средний ROUGE-2: 0.0317
Средний ROUGE-L: 0.1213


In [ ]:
#оценка трансформерной модели на задаче генерации текста с использованием ROUGE метрики rouge на distilgpt2

import os
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import evaluate
import numpy as np
from tqdm import tqdm


import sys
import os

# Получаем текущую рабочую директорию (где открыт ноутбук)
notebook_dir = os.getcwd()

# Путь к папке src, где лежит lstm_model.py
src_path = os.path.join(notebook_dir, "src")

# Загружаем тестовые тексты
project_root = r"C:\Users\Admin\NEIRO\SPR_2_FIN_PROJ\src"

# Путь к файлу с тестами
data_dir = os.path.join(project_root, "data", "processed")
test_file = os.path.join(data_dir, "test.txt")
# --- Функция для загрузки тестовых текстов ---
def load_texts(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

# Загружаем тестовые тексты
project_root = r"C:\Users\Admin\NEIRO\SPR_2_FIN_PROJ\src"

# Путь к файлу с тестами
data_dir = os.path.join(project_root, "data", "processed")
test_file = os.path.join(data_dir, "test.txt")
with open(test_file, "r", encoding="utf-8") as f:
    test_texts = [line.strip() for line in f if line.strip()]

# --- Загрузка модели и токенизатора ---
model_name = "distilgpt2" # на модели distilgpt2 rouge около 2% то есть меньше чем на LSTM поэтому переделал на GPT2
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# --- Pipeline для генерации ---
generator = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    device=-1,  # CPU; для GPU ставьте 0
)

# --- Метрика ROUGE ---
rouge = evaluate.load("rouge")

# --- Функция для генерации и оценки ---
def generate_and_evaluate_transformer(text):
    words = text.split()
    if len(words) < 5 or len(words) > 100:
        return None

    # Делим на 3/4 и 1/4
    split_idx = int(len(words) * 0.75)
    prompt_text = ' '.join(words[:split_idx])
    target_text = ' '.join(words[split_idx:])

    # Генерация продолжения
    out = generator(
        prompt_text,
        max_new_tokens=len(words) - split_idx,
        num_return_sequences=1,
        do_sample=False,
        top_p=0.95,
        temperature=1
    )

    generated_full = out[0]["generated_text"]
    # Берём только сгенерированное продолжение
    if generated_full.startswith(prompt_text):
        generated_part = generated_full[len(prompt_text):].strip()
    else:
        generated_part = generated_full

    # ROUGE
    rouge_scores = rouge.compute(predictions=[generated_part], references=[target_text])

    return {
        "original": text,
        "prompt": prompt_text,
        "generated_quarter": generated_part[:100],
        "target_quarter": target_text,
        "rouge1": rouge_scores["rouge1"],
        "rouge2": rouge_scores["rouge2"],
        "rougeL": rouge_scores["rougeL"]
    }

#  Вывод первых 10 примеров ---
print("\n=== ПЕРВЫЕ 10 ПРИМЕРОВ ===")
for i, text in tqdm(enumerate(test_texts[:10])):
    res = generate_and_evaluate_transformer(text)
    if res is None:
        continue

    print(f"\nПример {i+1}:")
    print(f"Оригинальный текст: {res['original']}")
    print(f"Промт (3/4 текста): {res['prompt']}")
    print(f"Сгенерировано 1/4 текста: {res['generated_quarter']}")
    print(f"Правильное продолжение: {res['target_quarter']}")
    print(f"ROUGE-1: {res['rouge1']:.4f}")
    print(f"ROUGE-2: {res['rouge2']:.4f}")
    print(f"ROUGE-L: {res['rougeL']:.4f}")
    print("-" * 80)

#  Средний ROUGE по 100 тестовым текстам ---
rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for text in tqdm(test_texts[:100], desc="Вычисление среднего ROUGE"):
    res = generate_and_evaluate_transformer(text)
    if res is None:
        continue
    rouge1_scores.append(res['rouge1'])
    rouge2_scores.append(res['rouge2'])
    rougeL_scores.append(res['rougeL'])

avg_rouge1 = np.mean(rouge1_scores) if rouge1_scores else 0.0
avg_rouge2 = np.mean(rouge2_scores) if rouge2_scores else 0.0
avg_rougeL = np.mean(rougeL_scores) if rougeL_scores else 0.0

print("\n=== СРЕДНИЙ ROUGE ПО 100 ТЕСТАМ ===")
print(f"Средний ROUGE-1: {avg_rouge1:.4f}")
print(f"Средний ROUGE-2: {avg_rouge2:.4f}")
print(f"Средний ROUGE-L: {avg_rougeL:.4f}")


Device set to use cpu



=== ПЕРВЫЕ 10 ПРИМЕРОВ ===


0it [00:00, ?it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
1it [00:00,  2.79it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Пример 1:
Оригинальный текст: she turn out fine at least she didn t annoy me doing malay hw at pm i don t know for what reason but i kinda miss hannah
Промт (3/4 текста): she turn out fine at least she didn t annoy me doing malay hw at pm i don t know for
Сгенерировано 1/4 текста: sure i dont know for sure i
Правильное продолжение: what reason but i kinda miss hannah
ROUGE-1: 0.1429
ROUGE-2: 0.0000
ROUGE-L: 0.1429
--------------------------------------------------------------------------------

Пример 2:
Оригинальный текст: ahhhhh so when are you leaving will you not make friday
Промт (3/4 текста): ahhhhh so when are you leaving will you
Сгенерировано 1/4 текста: be able to
Правильное продолжение: not make friday
ROUGE-1: 0.0000
ROUGE-2: 0.0000
ROUGE-L: 0.0000
--------------------------------------------------------------------------------


2it [00:00,  3.71it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
3it [00:00,  3.73it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


...


Пример 8:
Оригинальный текст: the car has to saty with dr toyota overnight
Промт (3/4 текста): the car has to saty with
Сгенерировано 1/4 текста: the car.
Правильное продолжение: dr toyota overnight
ROUGE-1: 0.0000
ROUGE-2: 0.0000
ROUGE-L: 0.0000
--------------------------------------------------------------------------------


9it [00:02,  4.49it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
10it [00:02,  4.73it/s]


Пример 9:
Оригинальный текст: i dreamt that i was failing one of my classes it was not fun
Промт (3/4 текста): i dreamt that i was failing one of my classes
Сгенерировано 1/4 текста: .
Правильное продолжение: it was not fun
ROUGE-1: 0.0000
ROUGE-2: 0.0000
ROUGE-L: 0.0000
--------------------------------------------------------------------------------

Пример 10:
Оригинальный текст: thanks i ll need it i hate packing
Промт (3/4 текста): thanks i ll need it i
Сгенерировано 1/4 текста: ll need
Правильное продолжение: hate packing
ROUGE-1: 0.0000
ROUGE-2: 0.0000
ROUGE-L: 0.0000
--------------------------------------------------------------------------------


10it [00:02,  4.29it/s]
Вычисление среднего ROUGE:   0%|          | 0/100 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Вычисление среднего ROUGE:   1%|          | 1/100 [00:00<00:26,  3.73it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Вычисление среднего ROUGE:   2%|▏         | 2/100 [00:00<00:22,  4.42it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Вычисление среднего ROUGE:   3%|▎         | 3/100 [00:00<00:22,  4.26it/s]The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFO


=== СРЕДНИЙ ROUGE ПО 100 ТЕСТАМ ===
Средний ROUGE-1: 0.1025
Средний ROUGE-2: 0.0332
Средний ROUGE-L: 0.1010



Вывод: любая трансформерная модель предпочительнее, чем LSTM  